# Week 49, Day 6: Forge: Week 49 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w049/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", -np.log2(1 / 32), "bits")
print("2.", round(2 * 0.6 / 1.6, 3))
print("3.", round(1 / (1 - 0.9), 1))

### Exercise: Part A: participation coefficient

Write participation(A, labels): for each node, count links into each module, divide by its degree and compute 1 minus the sum of squares. Apply it to the network with the four added hubs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2005)
M, n = 4, 30; N = M * n + 4
lab = np.r_[np.repeat(np.arange(M), n), [0, 1, 0, 0]]          # added hubs get a home module
A = np.zeros((N, N))
for i in range(M * n):
    for j in range(i + 1, M * n):
        if r.random() < (0.25 if lab[i] == lab[j] else 0.01):
            A[i, j] = A[j, i] = 1
hubs = np.arange(M * n, N)
for h, kind in zip(hubs, ["provincial", "provincial", "connector", "connector"]):
    if kind == "provincial":
        targets = r.choice(np.where(lab[: M * n] == lab[h])[0], 24, replace=False)
    else:
        targets = np.concatenate([r.choice(np.where(lab[: M * n] == m)[0], 6, replace=False) for m in range(M)])
    A[h, targets] = A[targets, h] = 1

def participation(A, labels):
    ...

k = A.sum(1); P = participation(A, lab)
for h, kind in zip(hubs, ["provincial", "provincial", "connector", "connector"]):
    print(f"node {h} ({kind}): degree {k[h]:.0f}, participation {P[h]:.2f}")
plt.scatter(P, k, s=10); plt.scatter(P[hubs], k[hubs], c="r"); plt.xlabel("participation coefficient"); plt.ylabel("degree"); plt.show()

In [ ]:
import numpy as np
def test_degree():
    "all four added nodes have degree above twice the median"
    assert all(k[h] > 2 * np.median(k) for h in hubs)
def test_classify():
    "connector hubs have P above 0.6 and provincial hubs below 0.3"
    assert all(P[h] > 0.6 for h in hubs[2:]) and all(P[h] < 0.3 for h in hubs[:2])

test_degree()
test_classify()
print("All checks passed")

### Exercise: Part B: instant revaluation

Compute the true values M R_new. For the SR agent, set R to the new reward vector after its run (it visits every state) and compute M R. For the model-free agent, start from the old values M R_old and apply TD(0) updates with alpha 0.1 for one run from state 0 to the end.

In [ ]:
n, gamma = 50, 0.9
T = np.zeros((n, n))
for s in range(n):
    T[s, min(s + 1, n - 1)] += 0.9; T[s, max(s - 1, 0)] += 0.1
Msr = np.linalg.inv(np.eye(n) - gamma * T)
R_old, R_new = np.zeros(n), np.zeros(n); R_old[45] = 1; R_new[10] = 1
V_true = Msr @ R_new
V_sr = ...
V_mf = Msr @ R_old                                # model-free values learned before the change
s = 0; g2 = np.random.default_rng(1)
while s < n - 1:
    s2 = min(s + 1, n - 1) if g2.random() < 0.9 else max(s - 1, 0)
    ...
    s = s2
print(f"correlation with true new values: SR {np.corrcoef(V_sr, V_true)[0, 1]:.3f}, model-free {np.corrcoef(V_mf, V_true)[0, 1]:.3f}")
plt.plot(V_true, label="true"); plt.plot(V_sr, "--", label="SR"); plt.plot(V_mf, label="model-free"); plt.legend(); plt.xlabel("state"); plt.show()

In [ ]:
import numpy as np
def test_sr():
    "after one run, the SR agent's values correlate above 0.99 with the true new values"
    assert np.corrcoef(V_sr, V_true)[0, 1] > 0.99
def test_mf():
    "the model-free agent's values still correlate below 0.5 with them"
    assert np.corrcoef(V_mf, V_true)[0, 1] < 0.5

test_sr()
test_mf()
print("All checks passed")